# Kaggle T4×2 DDP Validation — v2 (browser-safe + absolute paths)

Validates the FrozenOracle + HedgeMixer + SGD-TTT submission on Kaggle's free T4×2 runtime.

Exercises specifically:
- DDP NCCL across 2 ranks (only free environment we can do this in)
- The bucketed `dist.all_reduce` fix in TTT (collapses 100→1 NCCL launches)
- `FrozenNgramOracle.from_bytes` reload across 2 ranks (no per-rank temp files)
- The `'PGAR'` magic-prefixed artifact format roundtrip
- `index_put_(accumulate=True)` correctness on duplicate indices
- SGD TTT branch + HedgeMixer-with-oracle scoring

**Setup before running:**
1. Notebook → Settings → Accelerator → **GPU T4 x2**
2. Notebook → Settings → Internet → **On**
3. Add Data → Upload → upload `train_gpt.py` and `build_ngram_oracle.py` from the submission directory. The dataset will land at `/kaggle/input/<owner>/<dataset-name>/`. Update `UPLOAD_DIR` in cell 4 to match.

**v2 changes vs v1:** cell 7 redirects subprocess output to a file (no browser flood), uses an absolute path to `train_gpt.py` (cwd-independent), and adds NCCL flags Kaggle T4×2 needs (no NVLink, no IB).

In [ ]:
# 1. Environment check
!nvidia-smi -L
import torch, os
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'Devices: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} = {p.name}, {p.total_memory / 1024**3:.1f} GB')

In [ ]:
# 2. Install deps + clone the repo
!pip install -q zstandard sentencepiece
!git clone https://github.com/openai/parameter-golf.git /kaggle/working/pgolf 2>/dev/null || (cd /kaggle/working/pgolf && git pull)
%cd /kaggle/working/pgolf

In [ ]:
# 3. Download data — 1 shard is enough for the T4 sanity test
!python data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1
import glob, os
shards = sorted(glob.glob('/kaggle/working/pgolf/data/datasets/fineweb10B_sp1024/fineweb_train_*.bin'))
vals = sorted(glob.glob('/kaggle/working/pgolf/data/datasets/fineweb10B_sp1024/fineweb_val_*.bin'))
print(f'Train shards: {len(shards)}, Val shards: {len(vals)}')
for s in shards: print(f'  {s} ({os.path.getsize(s)/1e6:.0f} MB)')
for v in vals: print(f'  {v} ({os.path.getsize(v)/1e6:.0f} MB)')

In [ ]:
# 4. Copy submission files from the uploaded Kaggle dataset.
# Adjust UPLOAD_DIR to match your dataset name.
import shutil, os, glob

# Try a few common dataset paths
candidates = [
    '/kaggle/input/datasets/whatup45/oracle-submission-files1',
    '/kaggle/input/oracle-submission-files1',
    '/kaggle/input/oracle-submission-files',
    '/kaggle/input/oracle-submission',
]
# Also auto-detect any input dataset that contains train_gpt.py
for root in glob.glob('/kaggle/input/**/train_gpt.py', recursive=True):
    candidates.insert(0, os.path.dirname(root))

UPLOAD_DIR = next((c for c in candidates if os.path.exists(f'{c}/train_gpt.py')), None)
if UPLOAD_DIR is None:
    raise FileNotFoundError(
        f'Could not auto-locate train_gpt.py in /kaggle/input/. Tried: {candidates}.\n'
        'Either upload your files as a Kaggle dataset (Add Data -> Upload), or hard-code UPLOAD_DIR above.'
    )
print(f'Using UPLOAD_DIR = {UPLOAD_DIR}')

shutil.copy(f'{UPLOAD_DIR}/train_gpt.py', '/kaggle/working/pgolf/train_gpt.py')
shutil.copy(f'{UPLOAD_DIR}/build_ngram_oracle.py', '/kaggle/working/pgolf/build_ngram_oracle.py')

for p in ['train_gpt.py', 'build_ngram_oracle.py']:
    full = f'/kaggle/working/pgolf/{p}'
    print(f'  {p}: {"OK" if os.path.exists(full) else "MISSING"} ({os.path.getsize(full)} bytes)')

In [ ]:
# 5. FNV-1a NumPy/Torch equivalence self-test
!python /kaggle/working/pgolf/build_ngram_oracle.py --self-test

In [ ]:
# 6. Build a tiny oracle from a 10M-token slice of the first shard.
#    Full 80-shard build would take ~5 min on Kaggle CPU; 10M is enough to validate the pipeline.
import sys, io, struct, glob, time
sys.path.insert(0, '/kaggle/working/pgolf')
import numpy as np
from build_ngram_oracle import count_order, counts_to_int8_logprobs, BUCKET_CONFIGS, VOCAB, MAGIC, USE_ZSTD

try:
    import zstandard as zstd
except ImportError:
    import zlib

shard = sorted(glob.glob('/kaggle/working/pgolf/data/datasets/fineweb10B_sp1024/fineweb_train_*.bin'))[0]
header = np.fromfile(shard, dtype='<i4', count=256)
num_tokens = int(header[2])
tokens = np.fromfile(shard, dtype='<u2', count=min(num_tokens, 10_000_000), offset=256*4).astype(np.int32)
print(f'Loaded {len(tokens):,} tokens from {shard}')

t0 = time.time()
all_counts = {order: count_order(tokens, order, buckets) for order, buckets in BUCKET_CONFIGS.items()}
print(f'Counted in {time.time()-t0:.1f}s')

buf = io.BytesIO()
buf.write(struct.pack('<II', MAGIC, len(BUCKET_CONFIGS)))
for order in sorted(BUCKET_CONFIGS):
    data = counts_to_int8_logprobs(all_counts[order])
    rows, cols = data.shape
    buf.write(struct.pack('<III', order, rows, cols))
    buf.write(data.tobytes())
raw = buf.getvalue()
if USE_ZSTD:
    compressed = zstd.ZstdCompressor(level=22).compress(raw)
else:
    compressed = zlib.compress(raw, level=9)
with open('/kaggle/working/ngram_oracle.bin', 'wb') as f:
    f.write(compressed)
print(f'Oracle: {len(compressed):,} bytes ({len(compressed)/1024/1024:.2f} MB)')

In [ ]:
# 7. Run the full pipeline with DDP across 2 T4s.
#    - subprocess output -> file (browser-safe, doesn't crash on long log volume)
#    - absolute path to train_gpt.py (cwd-independent; fixes the 'No such file' crash)
#    - NCCL_P2P_DISABLE/IB_DISABLE for Kaggle T4×2 (no NVLink, no InfiniBand)
#    - Pre-flight check fails fast if oracle/data missing (kernel reset would wipe these)
import subprocess, os, time

PGOLF = '/kaggle/working/pgolf'
TRAIN_SCRIPT = f'{PGOLF}/train_gpt.py'
ORACLE = '/kaggle/working/ngram_oracle.bin'
DATA_DIR = f'{PGOLF}/data/datasets/fineweb10B_sp1024'

for label, path in [('train_gpt.py', TRAIN_SCRIPT), ('oracle', ORACLE), ('data dir', DATA_DIR)]:
    if not os.path.exists(path):
        raise SystemExit(f'MISSING: {label} at {path}. Re-run earlier cells (2, 3, 4, 6) first.')
    print(f'  OK  {label}: {path}')
print()

env = os.environ.copy()
env.update({
    'TORCHDYNAMO_DISABLE': '1',
    'SEED': '42',
    'NUM_LAYERS': '8', 'MODEL_DIM': '384', 'NUM_HEADS': '6', 'NUM_KV_HEADS': '6', 'MLP_HIDDEN': '1280',
    'TRAIN_SEQ_LEN': '512', 'TRAIN_BATCH_TOKENS': '32768', 'VAL_BATCH_SIZE': '32768', 'VOCAB_SIZE': '1024',
    'ITERATIONS': '200', 'WARMDOWN_ITERS': '60', 'WARMUP_STEPS': '10',
    'MAX_WALLCLOCK_SECONDS': '180',
    'VAL_LOSS_EVERY': '100', 'TRAIN_LOG_EVERY': '20',
    'VALUE_RESIDUAL': '1', 'GATED_ATTENTION': '1', 'EMA_ENABLED': '1', 'EMA_DECAY': '0.9985',
    'LATE_QAT': '0',
    'XSA_LAYERS': '4', 'ROPE_DIMS': '16', 'LN_SCALE': '1',
    'LEAKY_RELU_SLOPE': '0.75',
    'MUON_WD': '0.04', 'ADAM_WD': '0.04',
    'MATRIX_LR': '0.025', 'SCALAR_LR': '0.025', 'TIED_EMBED_LR': '0.035',
    'MUON_MOMENTUM': '0.99', 'MUON_MOMENTUM_WARMUP_START': '0.92', 'MUON_MOMENTUM_WARMUP_STEPS': '100',
    'TTT_ENABLED': '1', 'TTT_OPTIMIZER': 'sgd',
    'TTT_LR': '0.002', 'TTT_EPOCHS_PER_CHUNK': '2',
    'TTT_CHUNK_TOKENS': '16384', 'TTT_FREEZE_BLOCKS': '6', 'TTT_TIME_BUDGET': '90',
    'EVAL_STRIDE': '64',
    'BIGRAM_BUCKETS': '4096', 'BIGRAM_EMBED_DIM': '96',
    'CROWN_LAMBDA': '0.0',
    'HEDGE_ENABLED': '1', 'HEDGE_ETA': '0.01',
    'HEDGE_TRIGRAM_BUCKETS': '4096', 'HEDGE_CACHE_GAMMA': '0.999',
    'NGRAM_ORACLE_PATH': ORACLE,
    'RUN_ID': 'kaggle_t4x2_ddp_validation',
    # NCCL hints for Kaggle T4×2 (no NVLink between the two GPUs, no InfiniBand)
    'NCCL_P2P_DISABLE': '1',
    'NCCL_IB_DISABLE': '1',
})

LOG_FILE = '/kaggle/working/run_output.txt'
open(LOG_FILE, 'w').close()

# CRITICAL: pass an absolute path to train_gpt.py — torchrun resolves entrypoints
# against its own cwd, which on Kaggle can be /kaggle/working/ regardless of the
# subprocess cwd we set. Absolute path makes this irrelevant.
cmd = ['torchrun', '--standalone', '--nproc_per_node=2', TRAIN_SCRIPT]
print(f'Launching: {" ".join(cmd)}')
print(f'cwd: {PGOLF}')
print(f'Output -> {LOG_FILE}  (tailing every ~5s; browser-safe)\n')

t0 = time.time()
with open(LOG_FILE, 'w') as f_out:
    proc = subprocess.Popen(
        cmd, env=env, cwd=PGOLF,
        stdout=f_out, stderr=subprocess.STDOUT,
    )

interesting = (
    'oracle:', 'model_params', 'step:', 'warmup_step:', 'ema:', 'Serialized',
    'ttt_score_first:', 'ttt:done', 'final_int6', 'peak memory', 'WARN',
    'world_size', 'attention_mode', 'WARNING', 'Traceback', 'Error',
    "can't open", 'No such file', 'NCCL', 'CUDA',
)
seen_chunks = 0

with open(LOG_FILE) as f_in:
    while proc.poll() is None:
        time.sleep(5)
        for line in f_in.readlines():
            line = line.rstrip()
            if 'ttt_chunk:' in line:
                seen_chunks += 1
                if seen_chunks % 50 == 0:
                    print(line)
                continue
            if any(k in line for k in interesting):
                print(line)
        elapsed = time.time() - t0
        print(f'  [t={elapsed:.0f}s log={os.path.getsize(LOG_FILE)}B chunks={seen_chunks}]', flush=True)

    # Drain final lines after process exits
    for line in f_in.readlines():
        line = line.rstrip()
        if any(k in line for k in interesting) and 'ttt_chunk:' not in line:
            print(line)

print(f'\n=== Exit: {proc.returncode}, wall: {time.time()-t0:.1f}s ===')
print(f'Full log: {LOG_FILE} ({os.path.getsize(LOG_FILE):,} bytes)')

In [ ]:
# 8. Validate artifact: magic prefix, version, blob lengths, total size
import struct, os
artifact = '/kaggle/working/pgolf/final_model.int6.ptz'
if not os.path.exists(artifact):
    print('ERROR: artifact not produced — see cell 7 log')
else:
    size = os.path.getsize(artifact)
    print(f'Artifact: {size:,} bytes ({size/1024/1024:.2f} MB)')
    print(f'Budget remaining: {(16*1024*1024 - size)/1024/1024:.2f} MB')
    with open(artifact, 'rb') as f:
        head = f.read(16)
    magic, version, neural_len, oracle_len = struct.unpack('<IBxxxII', head)
    PGAR = 0x50474152
    magic_str = magic.to_bytes(4, 'little').decode('ascii', errors='replace')
    print(f"Magic: {magic:#010x} ('{magic_str}') {'OK' if magic == PGAR else 'FAIL — expected PGAR'}")
    print(f'Version: {version}')
    print(f'Neural blob: {neural_len:,} bytes ({neural_len/1024/1024:.2f} MB)')
    print(f'Oracle blob: {oracle_len:,} bytes ({oracle_len/1024/1024:.2f} MB)')
    print(f'Header + blobs total: {16 + neural_len + oracle_len:,} == {size:,}: {16+neural_len+oracle_len == size}')

In [ ]:
# 9. Final BPB summary — pull final_int6_zstd-22_roundtrip line from the log
import os, glob
logs = sorted(glob.glob('/kaggle/working/pgolf/logs/kaggle_t4x2_ddp_validation*.txt'),
              key=os.path.getmtime, reverse=True)
if logs:
    log = logs[0]
    print(f'Log: {log}')
    with open(log) as f:
        lines = f.readlines()
    for line in lines[-300:]:
        line = line.rstrip()
        if any(k in line for k in ('final_int6', 'val_bpb', 'peak memory', 'Serialized', 'ttt:done', 'oracle:loaded from artifact')):
            print(line)
else:
    print('No script log found — fallback to subprocess log:')
    with open('/kaggle/working/run_output.txt') as f:
        for line in f.readlines()[-100:]:
            line = line.rstrip()
            if any(k in line for k in ('final_int6', 'val_bpb', 'Serialized')):
                print(line)

## Expected output

**Cell 7** (~5-7 min):
```
Launching: torchrun --standalone --nproc_per_node=2 /kaggle/working/pgolf/train_gpt.py
[t=5s log=NB chunks=0]
world_size:2 grad_accum_steps:N
oracle:loaded for training orders=[1, 2, 3, 4, 5, 6, 7, 8] alpha=0.0
model_params:~12M
warmup_step:1/10 ... warmup_step:10/10
ema:init decay=0.9985
step:0/200 ... step:200/200 train_loss:~3.5
ema:applying EMA weights
Serialized model int6+zstd-22: neural=~6MB oracle=~3MB total=~9MB
oracle:loaded from artifact orders=[1, 2, 3, 4, 5, 6, 7, 8]
ttt_score_first:start lr=0.002 optimizer=sgd ... oracle=yes
(ttt_chunk lines every ~50)
ttt:done loss:... bpb:...
final_int6_zstd-22_roundtrip val_loss:... val_bpb:~1.6
=== Exit: 0, wall: ~400s ===
```

**Cell 8:**
```
Magic: 0x50474152 ('PGAR') OK
Version: 1
Neural blob: ~6 MB
Oracle blob: ~3 MB
Header + blobs total: ... == ...: True
```

## What this validates that nothing else can (free)

| Concern | Tested by | Coverage |
|---|---|---|
| FNV-1a NumPy/Torch hash equivalence | Cell 5 | YES |
| `index_put_(accumulate=True)` correctness on duplicates | Cell 7 (HedgeMixer.update_tables runs many times) | Implicit — runs without warning |
| **Bucketed `dist.all_reduce` in TTT** | **Cell 7 (DDP=2)** | **YES — only DDP environment we have** |
| **`FrozenNgramOracle.from_bytes` across ranks** | **Cell 7 (both ranks call it)** | **YES — no per-rank temp file** |
| Magic + version artifact wrapper | Cell 8 | YES — strict roundtrip |
| 16 MB cap | Cell 8 | YES — exact byte report |
| SGD TTT branch | Cell 7 (TTT_OPTIMIZER=sgd) | YES |
| Oracle in HedgeMixer at score time | Cell 7 (HEDGE_ENABLED=1) | YES |
| Competition-scale BPB | — | NO (would need 8×H100) |

What it does NOT validate:
- 11L/512d at 600s wall-clock
- 8-rank NCCL (we test 2-rank)
- A real BPB number against the leaderboard